# Build and run an MRRpy configuration — step by step

Run the next cell: it opens a **step-by-step form**, laid out like the QGIS plugin.

| Step | What you choose |
|---|---|
| **1 Terrain** | *I have a DEM file* → its path, **or** *Download a DEM* → the area (type it or draw it on the map), the dataset and resolution; then the basin outlet and the results folder |
| **2 Coordinate system** | the projection in metres (one click fills in your outlet's UTM zone) |
| **3 Precipitation** | *Design storm*, *Rain gauges* or *Satellite (IMERG)* — and only then Thiessen or IDW, the files or the storm date that choice needs |
| **4 Runoff** | *none / coefficient / raster / SCS curve number / physical*; for physical, tick the processes — each tick box opens its own settings |
| **5 Routing** | the routing method (its settings open underneath), simulation time, ground roughness, river channels, inflows |
| **6 Outputs and run** | gauges, maps, CPU/GPU — then **Check**, **Save** and **Run** |

Every choice opens its own settings **directly underneath it, only while it is selected**.
Rarely changed settings of a choice stay folded in a *More options* section inside its branch.

Prefer other tools? `MRRpy wizard` asks the same questions in a terminal,
`MRRpy init-config -o run.yaml` writes a commented file, and all of them share `run.yaml`.

**Install:** `pip install "MRRpy[notebook]"` (ipywidgets; the map picker uses ipyleaflet).

In [ ]:
from MRRpy import ConfigForm, explain, run_pipeline, plot_hydrograph

form = ConfigForm()          # or ConfigForm("run.yaml") to edit a saved configuration
form

## Google Earth Engine — only if you use it

Downloading a DEM, IMERG satellite rain, and the soil or land-cover maps come from
**Google Earth Engine**. You need a free Earth Engine account and a Google Cloud
**project ID** (e.g. `ee-yourname`; register one at https://code.earthengine.google.com/register).

Sign in **once per computer**: uncomment and run the cell below. If this computer has
never signed in, it prints a link — open it, sign in with your Google account, and paste
the code back into the box that appears (this works on a remote server too). After that,
the **Connect to Earth Engine** button in the form only checks that everything works.

In a terminal the same is: `MRRpy earth-engine-login --project ee-yourname`.

In [ ]:
import MRRpy
# MRRpy.connect_earth_engine("ee-yourname")   # ← put your project ID here

## Using the form

* Go through the steps with **Next** (or click a tab). *Back* returns to the previous step.
* Optional: choose a **Starting point** at the top and press *Apply* — it pre-selects a few
  settings — or **load** an existing file.
* Typed text is applied when you press **Enter** or leave the box. A message starting with
  **Problem:** appears under any value that can't be used.
* When a choice needs Google Earth Engine (DEM download, IMERG, soil or land-cover maps), a box
  for your Earth Engine project appears in that step.
* Step 6: **Check settings** lists anything that would stop a run, **Save** writes the YAML
  file, **Run the model** runs it here with a progress bar.

Keyboard only? **Tab** moves between fields and buttons, the **arrow keys** pick a radio option,
**Space** ticks a box. *Show an explanation under every setting* can be switched off.

## Carry on in code

The form is an ordinary Python object: read the configuration, check it, or turn it into code.

In [ ]:
print(form.summary())

problems = form.problems()
if problems:
    print("\nFix these before running:")
    for p in problems:
        print("  -", p)

In [ ]:
cfg = form.config            # a normal MRRpy Config
print(form.to_python())       # the same configuration as Python code

In [ ]:
form.save("my_run.yaml")     # the same as the Save button
# Later, from a terminal:  MRRpy run -c my_run.yaml

## Run the model

The *Run the model* button in step 6 does this too. While it runs, the line next to the
button says **Status: Running — 45 % done** and the model's log scrolls underneath. When it
ends it says **Status: Finished at 11:04:07 — took 23 s**, and a **Results** panel appears
with the peak flow, the water-balance check, the files written and a hydrograph plot.
The whole log is also saved as `mrrpy_run.log` in the results folder.

`hydrograph.csv` and `mass_balance.csv` are the last files a run writes. If they are in the
results folder, the run has finished.

In [ ]:
if form.problems():
    print("Not running yet: fix the problems listed above first.")
else:
    results = run_pipeline(form.config)
    plot_hydrograph(results)

In [ ]:
# Did the last run finish?  (Reads the results folder, so it also works for runs
# started from the button, from a cell or with `MRRpy run` in a terminal.)
import os
from MRRpy import plot_hydrograph

form.show_results()          # updates step 6: status, peak flow, files, hydrograph plot
print("Run state:", form.run_state)

hydrograph = form.config.HYDROGRAPH_CSV
if os.path.isfile(hydrograph):
    plot_hydrograph(form.config)   # the same plot as an ordinary notebook figure
else:
    print("No hydrograph yet in", os.path.abspath(form.config.OUTPUT_DIR))

## Explain any setting

`explain` takes a setting name, or a word to search for. (`MRRpy explain …` does the same in a terminal.)

In [ ]:
print(explain("CHANNEL_QBF_M3S"))

In [ ]:
print(explain("rain"))

## No ipywidgets? Answer questions instead

The text wizard needs nothing extra and also works inside a notebook. Uncomment and
run the cell below; it asks its questions one by one and saves `run.yaml`.

In [ ]:
# from MRRpy import run_wizard
# cfg = run_wizard()